Step 0: import dependencies and model

In [1]:
import numpy as np
import torch
import opt
from utils import post_proC, print_metrics, set_seed
from model import AllModel
from evaluation import eval
from load_data import load_data
import tqdm

/home/lcheng/FengyiZhou/venv/py38/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Step 1: set environment and load data

In [2]:
set_seed(seed=opt.args.seed)

X_omics1, X_omics2, adj_feature_omics1, adj_feature_omics2, label, adj_spatial_omics1, adj_spatial_omics2, a_omics1, data, raw_data = load_data()

opt.args.n_omics1 = X_omics1.shape[1]
opt.args.n_omics2 = X_omics2.shape[1]

if opt.args.name == 'Human_tonsil':
    opt.args.n_cluster = 7
    label = None
else:
    opt.args.n_cluster = len(np.unique(label))

Step 2: pretrain GAE

In [3]:
model = AllModel(X_omics2.shape[0]).cuda(opt.args.device)
optimizer0 = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=opt.args.pretrain_lr)
pbar = tqdm.tqdm(range(31), ncols=200)

for epoch in pbar:
    loss_rec = model(X_omics1, X_omics2, adj_feature_omics1, adj_feature_omics2, adj_spatial_omics1,
                         adj_spatial_omics2)

    pretrain_loss = loss_rec
    optimizer0.zero_grad()
    pretrain_loss.backward()
    optimizer0.step()

    pbar.set_postfix({'loss': '{0:1.4f}'.format(pretrain_loss)})

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 31/31 [00:04<00:00,  7.71it/s, loss=2.9150]


Step 3: pretrain GAE and fusion layers

In [4]:
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=opt.args.pretrain_lr)
pbar = tqdm.tqdm(range(opt.args.pretrain_epoch + 1), ncols=200)
for epoch in pbar:
    loss_rec = model.forward1(X_omics1, X_omics2, adj_feature_omics1, adj_feature_omics2, adj_spatial_omics1,
                         adj_spatial_omics2)

    pretrain_loss = loss_rec
    optimizer.zero_grad()
    pretrain_loss.backward()
    optimizer.step()

    pbar.set_postfix({'loss': '{0:1.4f}'.format(pretrain_loss)})


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 301/301 [00:40<00:00,  7.40it/s, loss=2.8154]


Step 4: train all model

In [5]:
optimizer2 = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=opt.args.train_lr)
pbar2 = tqdm.tqdm(range(opt.args.epoch + 1), ncols=200)
for epoch in pbar2:

    loss_rec, loss_selfExp, S = model.forward2(X_omics1, X_omics2, adj_feature_omics1, adj_feature_omics2,
                                                           adj_spatial_omics1, adj_spatial_omics2)

    total_loss =  loss_rec + loss_selfExp

    optimizer2.zero_grad()
    total_loss.backward()
    optimizer2.step()

    pbar2.set_postfix({'loss': '{0:1.4f}'.format(total_loss)})
    if epoch % opt.args.epoch == 0 and epoch != 0:
        S_cpu = S.cpu().detach().numpy()
        pred, _ = post_proC(S_cpu, opt.args.n_cluster)

        if label is not None:
            metrics = eval(label, pred)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉| 2000/2001 [28:09<00:00,  1.16it/s, loss=8.3124]


our         (ACC): 0.515786
our         jaccard: 0.263812


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2001/2001 [28:27<00:00,  1.17it/s, loss=8.3124]

our         F_measure: 0.417486
our         Mutual Information: 0.675445
Our         (NMI): 0.375941
Our         (AMI): 0.372049
Our         V-measure: 0.375941
Our         Homogeneity: 0.354314 Completeness: 0.400381
Our         (ARI): 0.281091
Our         (FMI): 0.423502
